# Building Agents with Google's MCP Servers

## Overview

In this notebook, you will build an AI Agent using the Agent Development Kit (ADK) and Model Context Protocol (MCP) servers. You'll learn how to leverage MCP servers to connect your agent to external data sources and tools, specifically Google BigQuery and Google Maps, to solve a complex business problem. <br>
Throughout this notebook, you will set up the environment, interact with MCP servers directly to understand their protocol, and finally build and test an agent that uses these servers via the ADK.

### Scenario: Launching a Bakery

To apply these concepts, you will build an agent to help a friend launch a new high-end sourdough bakery in Los Angeles. The agent will need to:
1. Query BigQuery to find macro trends (demographics, competitor pricing, sales history).
2. Use Google Maps to validate micro-location details.

## Learning Goals

By the end of this notebook, you will understand how to:
* Understand the concept of **Model Context Protocol (MCP)** and how it enables standardized tool integration.
* Interact with MCP servers directly.
* Create an ADK agent that uses MCP servers as tools.
* Use the **ADK Developer UI** to interact with and inspect the agent's execution flow involving MCP tools.

## Setup

In this section, we will set up the necessary data in BigQuery. We will create a dataset named `mcp_bakery` and load sample data into four tables:
- `demographics`
- `bakery_prices`
- `sales_history_weekly`
- `foot_traffic`

Instead of running an external script, we will define the setup steps directly in this notebook.

In [ ]:
import os
import pandas as pd
import google.auth
from google.cloud import bigquery
from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams
import time
import json
import requests


In [ ]:
# Install gcloud alpha component if needed
!gcloud components install alpha --quiet

# Enable necessary APIs
!gcloud services enable mapstools.googleapis.com
!gcloud services enable bigquery.googleapis.com

# Enable MCP for services
!gcloud beta services mcp enable mapstools.googleapis.com
!gcloud beta services mcp enable bigquery.googleapis.com

### Create Google Maps API Key

We will attempt to automatically create a restricted API key for the Google Maps MCP service. We use Python to parse the JSON response from the `gcloud` command and set it in the `MAPS_API_KEY` environment variable.

In [ ]:
# Run gcloud command and capture output
api_key_name = f"bakery-demo-key-{int(time.time())}"
result = !gcloud alpha services api-keys create --display-name="{api_key_name}" --api-target=service=mapstools.googleapis.com --format=json --quiet

result_str = "\n".join(result)

# Find the first '{' to skip potential non-JSON text
start = result_str.find("{")

# Decode the first JSON object found
decoder = json.JSONDecoder()
key_data, idx = decoder.raw_decode(result_str[start:])

api_key = key_data.get('keyString')
os.environ['MAPS_API_KEY'] = api_key

### Use Data from Google Cloud Storage

We will use the data stored in the public bucket `gs://asl-public-data/data/mcp-bakery`. We will load this data directly into BigQuery in the next step.

In [ ]:
!gcloud storage ls gs://asl-public-data/data/mcp-bakery/

### Create BigQuery Dataset and Tables

Now we will create the BigQuery dataset and tables, and load the data from the CSV files we just created.

In [ ]:
credentials, project_id = google.auth.default()
if not project_id:
    project_id = PROJECT_ID

client = bigquery.Client(project=project_id, credentials=credentials)

dataset_id = f"{project_id}.mcp_bakery"
dataset = bigquery.Dataset(dataset_id)
dataset.location = "US"

try:
    dataset = client.create_dataset(dataset, timeout=30)
    print(f"Created dataset {dataset_id}")
except Exception as e:
    print(f"Dataset might already exist or error: {e}")

# Helper function to load CSV from GCS to BQ
def load_csv_from_gcs_to_bq(table_name, schema):
    table_id = f"{dataset_id}.{table_name}"
    uri = f"gs://asl-public-data/data/mcp-bakery/{table_name}.csv"
    
    job_config = bigquery.LoadJobConfig(
        schema=schema,
        skip_leading_rows=1,
        source_format=bigquery.SourceFormat.CSV,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    )
    
    job = client.load_table_from_uri(uri, table_id, job_config=job_config)
    job.result()  # Wait for the job to complete.
    print(f"Loaded rows into {table_id} from {uri}.")

# Schemas
demo_schema = [
    bigquery.SchemaField("zip_code", "STRING"),
    bigquery.SchemaField("city", "STRING"),
    bigquery.SchemaField("neighborhood", "STRING"),
    bigquery.SchemaField("total_population", "INT64"),
    bigquery.SchemaField("median_age", "FLOAT64"),
    bigquery.SchemaField("bachelors_degree_pct", "FLOAT64"),
    bigquery.SchemaField("foot_traffic_index", "FLOAT64"),
]

prices_schema = [
    bigquery.SchemaField("store_name", "STRING"),
    bigquery.SchemaField("product_type", "STRING"),
    bigquery.SchemaField("price", "FLOAT64"),
    bigquery.SchemaField("region", "STRING"),
    bigquery.SchemaField("is_organic", "BOOLEAN"),
]

sales_schema = [
    bigquery.SchemaField("week_start_date", "DATE"),
    bigquery.SchemaField("store_location", "STRING"),
    bigquery.SchemaField("product_type", "STRING"),
    bigquery.SchemaField("quantity_sold", "INT64"),
    bigquery.SchemaField("total_revenue", "FLOAT64"),
]

traffic_schema = [
    bigquery.SchemaField("zip_code", "STRING"),
    bigquery.SchemaField("time_of_day", "STRING"),
    bigquery.SchemaField("foot_traffic_score", "FLOAT64"),
]

# Load tables
load_csv_from_gcs_to_bq("demographics", demo_schema)
load_csv_from_gcs_to_bq("bakery_prices", prices_schema)
load_csv_from_gcs_to_bq("sales_history_weekly", sales_schema)
load_csv_from_gcs_to_bq("foot_traffic", traffic_schema)

### Verify Data in BigQuery

Let's check the first few rows of each table to understand what kind of data we have and verify that it was loaded correctly.

In [ ]:
# Function to query and display head of a table
def show_table_head(table_name):
    query = f"SELECT * FROM `{project_id}.mcp_bakery.{table_name}` LIMIT 5"
    df = client.query(query).to_dataframe()
    print(f"\n--- Table: {table_name} ---")
    display(df)

for table in ["demographics", "bakery_prices", "sales_history_weekly", "foot_traffic"]:
    show_table_head(table)

## Understand MCP Servers

### What is MCP?
Model Context Protocol (MCP) is an open standard that enables AI models to securely interact with external data sources and tools. Instead of writing custom integration code for every tool, MCP provides a standardized way for models to discover and use capabilities provided by MCP servers.

Google provides [a lot of Google-hosted managed MCP servers](https://docs.cloud.google.com/mcp/supported-products).

In this tutorial, we will use two MCP servers:
1. [**BigQuery MCP**](https://docs.cloud.google.com/bigquery/docs/reference/mcp): Allows the agent to query BigQuery datasets.
2. [**Google Maps MCP**](https://developers.google.com/maps/ai/grounding-lite/reference/mcp): Allows the agent to interact with Google Maps APIs (geocoding, places, etc.).

### Calling MCP Server Directly

To understand how MCP works under the hood, we can call the MCP server directly using standard HTTP requests. MCP servers typically use JSON-RPC over HTTP or SSE.

Here we make a direct POST request to list the available tools.

In [ ]:
MAPS_MCP_URL = "https://mapstools.googleapis.com/mcp"
BIGQUERY_MCP_URL = "https://bigquery.googleapis.com/mcp"

Let's setup the header.

In [ ]:
# Direct call to BigQuery MCP server to list tools
credentials, project_id = google.auth.default(
)
credentials.refresh(google.auth.transport.requests.Request())
oauth_token = credentials.token

headers = {
    "Authorization": f"Bearer {oauth_token}",
    "x-goog-user-project": project_id,
    "Content-Type": "application/json",
}


Let's list available tools under the BigQuery MCP.

In [ ]:

# MCP list tools request payload
payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/list",
    "params": {}
}

response = requests.post(BIGQUERY_MCP_URL, headers=headers, json=payload)

resp_json = response.json()
tools = resp_json.get('result', {}).get('tools', [])
print("Available tools:")
for t in tools:
    print(f"- {t.get('name')}")

Now let's try to call a specific tool directly. We will call the `execute_sql` tool to query the `demographics` table we created earlier. We will only print the main content of the response to avoid long output.

In [ ]:
payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {
        "name": "execute_sql",
        "arguments": {
            "project_id": project_id,
            "query": f"SELECT * FROM `{project_id}.mcp_bakery.demographics` LIMIT 2"
        }
    }
}

response = requests.post(BIGQUERY_MCP_URL, headers=headers, json=payload)

resp_json = response.json()
result = resp_json.get('result', {})
content = result.get('content', [])

print("Result Content:")
print(content)

Also, let's check what kind of tools are available under the Google Map MCP.

In [ ]:
payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/list",
    "params": {}
}

response = requests.post(MAPS_MCP_URL, headers=headers, json=payload)

resp_json = response.json()
tools = resp_json.get('result', {}).get('tools', [])
print("Available tools:")
for t in tools:
    print(f"- {t.get('name')}")

## Define the Agent with MCP

To use the ADK Developer UI, we need to save our agent definition to a directory. We will create a directory named `mcp_bakery_agents` and write `tools.py` and `agent.py` files containing the agent and toolset definitions.

In [ ]:
os.makedirs('mcp_bakery_agents/agent', exist_ok=True)

with open('mcp_bakery_agents/.env', 'w') as f:
    f.write(f"MAPS_API_KEY={os.environ.get('MAPS_API_KEY', '')}\n")
    f.write(f"GOOGLE_CLOUD_LOCATION=us-central1\n")
    f.write(f"GOOGLE_GENAI_USE_VERTEXAI=TRUE\n")
    f.write(f"PROJECT_ID={project_id}\n")

In [ ]:
%%writefile mcp_bakery_agents/agent/tools.py
import os
import dotenv
import google.auth
from google.adk.tools.mcp_tool.mcp_toolset import MCPToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams

MAPS_MCP_URL = "https://mapstools.googleapis.com/mcp"
BIGQUERY_MCP_URL = "https://bigquery.googleapis.com/mcp"

def get_maps_mcp_toolset():
    maps_api_key = os.getenv('MAPS_API_KEY', 'YOUR_MAPS_API_KEY')
    return MCPToolset(
        connection_params=StreamableHTTPConnectionParams(
            url=MAPS_MCP_URL,
            headers={"X-Goog-Api-Key": maps_api_key},
            timeout=30.0,
            sse_read_timeout=300.0
        )
    )

def get_bigquery_mcp_toolset():
    credentials, project_id = google.auth.default(
        scopes=["https://www.googleapis.com/auth/bigquery"]
    )
    credentials.refresh(google.auth.transport.requests.Request())
    oauth_token = credentials.token
    return MCPToolset(
        connection_params=StreamableHTTPConnectionParams(
            url=BIGQUERY_MCP_URL,
            headers={
                "Authorization": f"Bearer {oauth_token}",
                "x-goog-user-project": project_id
            },
            timeout=30.0,
            sse_read_timeout=300.0
        )
    )

In [ ]:
%%writefile mcp_bakery_agents/agent/agent.py
import os
from google.adk.agents import LlmAgent
from . import tools

bq_toolset = tools.get_bigquery_mcp_toolset()
maps_toolset = tools.get_maps_mcp_toolset()

PROJECT_ID = os.getenv('PROJECT_ID')

PROMPT = f"""
You are a helpful business consultant helping a friend launch a new high-end sourdough bakery in Los Angeles.
Your goal is to recommend the best neighborhood and a premium price point.

To do this, you should:
1. Query BigQuery mcp_bakery dataset under {PROJECT_ID} project to find neighborhoods with high bachelors degree percentage and high foot traffic index (macro trends).
2. Query BigQuery mcp_bakery dataset under {PROJECT_ID} project to analyze competitor pricing for 'Sourdough Loaf' in 'Los Angeles Metro' to suggest a premium price.
3. Use Google Maps tools to validate micro-location details or search for specific locations in the chosen neighborhood.

Google Cloud project ID:
{PROJECT_ID}

Be strategic and combine insights from both sources.
"""

root_agent = LlmAgent(
    model="gemini-2.5-flash",
    name="bakery_consultant_agent",
    instruction=PROMPT,
    tools=[bq_toolset, maps_toolset],
)

## Call Agent via ADK Web

You can launch the ADK Developer UI to interact with your agents using the `adk web` command.

Execute the appropriate cell below depending on your environment.

Once the UI is running, you can interact with the agent. Try asking questions like:

- "Which zip code in Los Angeles has the highest morning foot traffic?"
- "Is the density of existing bakeries too high in the Silver Lake area?"
- "Based on local competitor pricing, what premium price can I charge for a sourdough loaf?"
- "Forecast the sales performance based on historical trends."

In [ ]:
# On Cloud Workstations
!adk web mcp_bakery_agents --allow_origins "regex:https://.*\.cloudworkstations\.dev"

In [ ]:
# Note: if you are using Vertex AI Workbench, remove the comment out and run the cell below.
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# echo "--------------------------------------------------------"
# echo "🔗 ACCESS HERE: https://${PROXY_BASE}/proxy/8000"
# echo "--------------------------------------------------------"
# adk web mcp_bakery_agents --url_prefix /proxy/8000  --allow_origins "regex:https://.*\.notebooks\.googleusercontent\.com"

## Summary

In this notebook, we demonstrated how to:
1. Set up sample data in BigQuery for a business scenario.
2. Connect to Google-hosted MCP servers for BigQuery and Google Maps using ADK.
3. Define an AI Agent that leverages these MCP servers to solve complex business problems.
4. Launch the ADK Developer UI to interact with the agent.

**Citation & Disclaimer**

This notebook is based on and updated from the public example: [Google MCP Launch My Bakery](https://github.com/google/mcp/tree/main/examples/launchmybakery).

Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.